# 02 — Feature Engineering (Fake Review Signals)


In [5]:
import pandas as pd
df = pd.read_csv("/content/Review-Guardian/data/processed/cleaned_data.csv")
print(df.shape)
df.columns.tolist()

(6232, 25)


['s.no',
 'helpfulVoteCount',
 'images/0',
 'images/1',
 'images/2',
 'images/3',
 'images/4',
 'images/5',
 'images/6',
 'images/7',
 'productASIN',
 'productVariant',
 'rating',
 'reviewID',
 'reviewMetadata',
 'reviewPosition',
 'reviewText',
 'reviewTitle',
 'reviewURL',
 'verifiedPurchase',
 'videos/0',
 'cleaned_review_text',
 'sentiment_score',
 'at',
 'content_clean']

Build each heuristic feature

Duplicate / near-duplicate content (spammy accounts often reuse text):

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# dropping rows with missing content_clean values
df = df.dropna(subset=["content_clean"])
df = df[df["content_clean"].str.strip() != ""]

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['content_clean'])

# Exact duplicate flag (fast)
df['is_exact_dup'] = df.duplicated(subset=['content_clean'], keep=False).astype(int)

# Near-duplicate flag via cosine similarity (do this in batches if dataset is large — full pairwise
# similarity is O(n^2) and will blow up memory past ~20k rows; sample or use nearest-neighbors instead)
from sklearn.neighbors import NearestNeighbors
n_neighbors = min(2, X.shape[0])
nn = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine').fit(X)
distances, indices = nn.kneighbors(X)
df['near_dup_score'] = 1 - distances[:, -1]   # similarity to nearest OTHER review
df['is_near_dup'] = (df['near_dup_score'] > 0.9).astype(int)

Burstiness (spike in review volume in a short window):

In [7]:
print(df["at"].dtype)

object


In [8]:
df["at"] = pd.to_datetime(df["at"], errors="coerce")
daily_counts = df.groupby(df['at'].dt.date).size()
mean_c, std_c = daily_counts.mean(), daily_counts.std()
burst_days = daily_counts[daily_counts > mean_c + 2*std_c].index  # >2 std above average
df['is_burst_day'] = df['at'].dt.date.isin(burst_days).astype(int)

~~One-hit-wonder accounts~~ — **dropped**: this dataset has no user identifier column (no `userName`/`user_id`), so this signal cannot be computed. Do not fabricate it from `reviewID` (every review already has a unique `reviewID` by construction, which would flag 100% of rows as "one-hit-wonder" and add pure noise).

Rating deviation from consensus (per **product**, using the real `productASIN` — this is more correct than the old app-version proxy):

In [9]:
product_mean = df.groupby('productASIN')['rating'].transform('mean')
df['rating_deviation'] = (df['rating'] - product_mean).abs()

Verified purchase signal (new — not available in the old app-review dataset). Unverified purchases are a classic fake-review red flag:

In [10]:
df['is_unverified'] = (df['verifiedPurchase'] == False).astype(int)

Linguistic features

In [11]:
df['exclamation_count'] = df['reviewText'].str.count('!')
df['word_count'] = df['content_clean'].str.split().str.len()
df['avg_word_len'] = df['content_clean'].apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
df['superlative_count'] = df['content_clean'].str.count(r'\b(best|worst|amazing|terrible|perfect|awful)\b')

Combine into one pseudo-label

In [12]:
# Normalize each signal to 0-1, then take a weighted sum. Weights are a judgment call —
# document why you picked them in your report.
# NOTE: weights re-distributed after dropping is_one_hit_wonder (no user id in this dataset)
# and adding is_unverified (new signal available in this dataset).
from sklearn.preprocessing import MinMaxScaler

signals = ['is_exact_dup', 'is_near_dup', 'is_burst_day', 'is_unverified', 'rating_deviation']
scaler = MinMaxScaler()
df[signals] = scaler.fit_transform(df[signals])

weights = {'is_exact_dup': 0.3, 'is_near_dup': 0.25, 'is_burst_day': 0.2,
           'is_unverified': 0.15, 'rating_deviation': 0.1}
df['fake_score'] = sum(df[s] * w for s, w in weights.items())

# Threshold into binary pseudo_label — inspect the distribution before picking the cutoff
print(df['fake_score'].describe())
df['pseudo_label'] = (df['fake_score'] > df['fake_score'].quantile(0.85)).astype(int)  # top 15% flagged as "likely fake"
print(df['pseudo_label'].value_counts())

count    6232.000000
mean        0.120638
std         0.113096
min         0.000000
25%         0.011429
50%         0.080000
75%         0.211429
max         0.791270
Name: fake_score, dtype: float64
pseudo_label
0    5479
1     753
Name: count, dtype: int64


Deliverable

In [13]:
feature_cols = ['reviewText', 'content_clean', 'rating', 'productASIN', 'is_exact_dup', 'is_near_dup', 'is_burst_day',
                'is_unverified', 'rating_deviation', 'exclamation_count', 'word_count',
                'avg_word_len', 'superlative_count', 'fake_score', 'pseudo_label']
df[feature_cols].to_csv('/content/Review-Guardian/data/processed/labeled_reviews.csv', index=False)
print('Saved:', df[feature_cols].shape)

Saved: (6232, 15)
